# 🚀 Google Colab: ฝึกสอนโมเดล YOLOv11 สำหรับนับวัตถุ (Nut & Bolt Counter)

สมุดโค้ดนี้ใช้สำหรับการเทรนโมเดล **Ultralytics YOLOv11** ด้วย **GPU ของ Google ฟรี**
เมื่อเทรนเสร็จ ระบบจะส่งออกไฟล์ **`best.onnx`** และดาวน์โหลดลงเครื่องคอมพิวเตอร์ของคุณอัตโนมัติ เพื่อนำไปลากใส่หน้าเว็บ HTML5 ใช้นับสดได้ทันที!

### ขั้นตอนที่ 1: ตรวจสอบการ์ดจอ GPU (ฟรีจาก Google)
*(ไปที่เมนู Runtime -> Change runtime type -> เลือก T4 GPU ก่อนเริ่ม)*

In [ ]:
!nvidia-smi

### ขั้นตอนที่ 2: ติดตั้งไลบรารี Ultralytics YOLOv11

In [ ]:
!pip install -q ultralytics onnx onnxslim

### ขั้นตอนที่ 3: เตรียมชุดข้อมูล (Dataset)
คุณสามารถเลือกทำอย่างใดอย่างหนึ่ง:
- **วิธี A (สำหรับทดสอบทันที):** รันโค้ดด้านล่างเพื่อสร้างภาพตัวอย่างน็อตอัตโนมัติ 50 ภาพ
- **วิธี B (ใช้ภาพของคุณจริง):** อัปโหลดไฟล์ `dataset.zip` ของคุณขึ้นมาแล้วแตกไฟล์

In [ ]:
import os, time, cv2
import numpy as np
from pathlib import Path

# สร้างโครงสร้างโฟลเดอร์สำหรับ YOLO
dataset_dir = Path("/content/dataset")
img_dir = dataset_dir / "images" / "train"
lbl_dir = dataset_dir / "labels" / "train"
img_dir.mkdir(parents=True, exist_ok=True)
lbl_dir.mkdir(parents=True, exist_ok=True)

# สร้างภาพตัวอย่างน็อตจำลอง (50 ตัวอย่าง)
print("กำลังสร้างภาพตัวอย่างน็อตสำหรับเทรน...")
for i in range(50):
    w_img, h_img = 640, 640
    bg = np.full((h_img, w_img, 3), np.random.randint(230, 250), dtype=np.uint8)
    labels = []
    for _ in range(np.random.randint(1, 4)):
        nx, ny = np.random.randint(100, 540), np.random.randint(100, 540)
        r = np.random.randint(28, 45)
        ang = np.random.uniform(0, np.pi)
        pts = np.array([[int(nx + r * np.cos(ang + k * np.pi / 3)), int(ny + r * np.sin(ang + k * np.pi / 3))] for k in range(6)], np.int32)
        cv2.fillPoly(bg, [pts], (40, 40, 40))
        cv2.circle(bg, (nx, ny), int(r * 0.45), (240, 240, 240), -1)
        
        bx1, by1 = max(0, nx - r), max(0, ny - r)
        bx2, by2 = min(w_img, nx + r), min(h_img, ny + r)
        cx, cy = (bx1 + bx2) / 2 / w_img, (by1 + by2) / 2 / h_img
        bw, bh = (bx2 - bx1) / w_img, (by2 - by1) / h_img
        labels.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        
    cv2.imwrite(str(img_dir / f"nut_{i}.jpg"), bg)
    with open(lbl_dir / f"nut_{i}.txt", "w") as f:
        f.write("\n".join(labels))

# สร้างไฟล์ data.yaml
yaml_content = f"""path: /content/dataset
train: images/train
val: images/train
names:
  0: nut
"""
with open("/content/data.yaml", "w") as f:
    f.write(yaml_content)

print("✅ เตรียม Dataset สำเร็จ! พร้อมสำหรับการเทรนแล้ว")

### ขั้นตอนที่ 4: สั่งเทรนโมเดล YOLOv11 บน GPU
*(ใช้โมเดล YOLO11 Nano: yolo11n.pt ซึ่งเล็ก เร็ว และเหมาะกับการรันบนหน้าเว็บที่สุด)*

In [ ]:
from ultralytics import YOLO

# โหลดโมเดลฐาน YOLOv11 Nano
model = YOLO("yolo11n.pt")

# เริ่มการเทรน 30 รอบ (Epochs)
results = model.train(
    data="/content/data.yaml",
    epochs=30,
    batch=16,
    imgsz=640,
    device=0,
    project="/content/runs",
    name="bolt_nut_detector",
    exist_ok=True
)

print("🎉 การฝึกสอนโมเดล YOLOv11 เสร็จสมบูรณ์แล้ว!")

### ขั้นตอนที่ 5: แปลงโมเดลเป็น ONNX (.onnx) สำหรับใช้งานบน HTML5

In [ ]:
# ส่งออกเป็น ONNX
trained_model = YOLO("/content/runs/bolt_nut_detector/weights/best.pt")
onnx_path = trained_model.export(format="onnx", imgsz=640, dynamic=False)
print(f"✅ สร้างไฟล์ ONNX สำเร็จที่: {onnx_path}")

### ขั้นตอนที่ 6: ดาวน์โหลดไฟล์ `best.onnx` ลงเครื่องคอมพิวเตอร์ของคุณ
*(เมื่อดาวน์โหลดแล้ว ให้นำไฟล์นี้ไปลากใส่หน้าเว็บ HTML5 เพื่อเริ่มนับได้ทันที)*

In [ ]:
from google.colab import files
files.download(onnx_path)